# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding A: Higher-competition keywords get fewer clicks
- **Label Source:** `clicks_90d` is directly observed behavioral data.
- **Validation Claim:** The original claim was cross-sectional. It supports an association rather than causality. We must check if `competition_level` is confounded with `content_type`.

### Finding B: Model underperformance under grouped split
- **Label Source:** `ctr` (Click-Through Rate) is directly observed.
- **Validation Claim:** The claim is carried by the design *because* the grouped split prevents identity leakage of clients, providing a realistic estimate of performance on new clients.

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import os

# Load data using the pattern from w03
DATA_PATH = 'my-ml-starter/data/raw/content_refresh_anonymized.csv'
if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
else:
    # Fallback/placeholder for local execution environment consistency
    print(f'Warning: {DATA_PATH} not found. Ensure file is in repo.')
    df = pd.DataFrame() # Placeholder

In [5]:
# Audit check for Finding A
if not df.empty:
    audit_tab = pd.crosstab(df['competition_level'], df['content_type'], normalize='index')
    display(audit_tab)
    print("\nCrosstab shows distribution of content types across competition levels.")

content_type,comparison article,keyword article
competition_level,,
HIGH,0.000000,1.000000
LOW,0.030442,0.969558
MEDIUM,0.000000,1.000000



Crosstab shows distribution of content types across competition levels.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

The random split (Before) yields optimistic metrics due to data leakage where the same client appears in both sets. The grouped split (After) is the honest measure of generalization.

In [9]:
if not df.empty:
    # EXACT SAME signal_features and preprocessing from w05_model.ipynb
    signal_features = [
        'word_count', 'char_count', 'content_age_days', 'days_since_last_update',
        'content_type', 'main_intent', 'competition_level', 'search_volume',
        'competition', 'cpc', 'word_count_tier', 'age_tier', 'freshness_tier'
    ]
    target = 'ctr'

    # Preprocessing: Handle missing values as per w05
    X_raw = df[signal_features].copy()

    numeric_cols = X_raw.select_dtypes(include=[np.number]).columns
    categorical_cols = X_raw.select_dtypes(exclude=[np.number]).columns

    X_raw[numeric_cols] = X_raw[numeric_cols].fillna(-1)
    X_raw[categorical_cols] = X_raw[categorical_cols].fillna('unknown')

    X = pd.get_dummies(X_raw)
    y = df[target]
    groups = df['client_id']

    def evaluate(X_tr, X_te, y_tr, y_te):
        model = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_te)
        return mean_absolute_error(y_te, preds), r2_score(y_te, preds)

    # (a) BEFORE: Random Split
    X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.2, random_state=42)
    mae_r, r2_r = evaluate(X_train_r, X_test_r, y_train_r, y_test_r)

    # (b) AFTER: Grouped Split
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups))
    X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
    y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]
    mae_g, r2_g = evaluate(X_train_g, X_test_g, y_train_g, y_test_g)

    results = pd.DataFrame({
        'Metric': ['MAE', 'R2'],
        'Before (Random Split, Leaky)': [mae_r, r2_r],
        'After (Grouped Split, Honest)': [mae_g, r2_g]
    })
    display(results)

,Metric,"Before (Random Split, Leaky)","After (Grouped Split, Honest)"
0,MAE,0.678293,0.873754
1,R2,0.079780,-1.885138


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [8]:
if not df.empty:
    leaky_cols = ['avg_position', 'position_tier', 'impression_tier', 'trend_direction',
                  'trend_pct', 'ctr', 'clicks_90d', 'impressions_90d', 'sessions_90d', 'engagement_rate']
    used_cols = X.columns.tolist()

    print("Columns used in model:", used_cols)
    found_leakage = [col for col in leaky_cols if col in used_cols]

    assert len(found_leakage) == 0, f"Leakage detected: {found_leakage}"
    print("\nSuccess: No leakage columns found in feature set.")

Columns used in model: ['competition', 'search_volume', 'content_age_days', 'word_count']

Success: No leakage columns found in feature set.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Rewritten Claim
In this 90-day snapshot, pages targeting higher-competition keywords showed fewer clicks on average (Mean log1p(clicks_90d): LOW 1.34 n=22,896; MEDIUM 1.08 n=1,836; HIGH 0.95 n=2,658). This is an observed association, not a causal claim, since competition_level was not randomly assigned and may be confounded with content type.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.